# 1 — Flyback Converter Modeling

> **Goal.** Derive the small-signal model of the flyback (an **isolated
> buck-boost**) from scratch, show how the transformer's turns ratio
> enters the math as a single scaling factor, and identify the same
> RHP zero as the buck-boost — pushed UP in frequency by the
> reflection.

**Prerequisites**

- Buck-boost modeling notebook (`projects/converters/buck_boost/`).
  The flyback's small-signal model maps directly onto the buck-boost's
  via reflected impedance; this notebook formalizes that mapping.
- Basic transformer concept: $v_s = n \\cdot v_p$, $i_s = i_p / n$.

**What you'll be able to do at the end**

1. Identify the flyback on a schematic and distinguish it from a
   forward / boost / non-isolated buck-boost.
2. Apply state-space averaging to the flyback's two intervals (ON
   stores energy on the primary, OFF releases it to the secondary).
3. Use the **reflection trick** — analyze on the primary side as a
   buck-boost, then multiply by $n$ — and show it produces the same
   answer as direct analysis.
4. Locate the RHP zero and explain why it's at a HIGHER frequency
   than the buck-boost at the same $L_m, C, R, D$ (the $1/n^2$
   reflection of the secondary load).


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from flyback_model import (
    FlybackParams,
    flyback_state_space,
    control_to_output_tf,
    line_to_output_tf,
    output_impedance_tf,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The flyback topology

```
   V_g+ ----- S -----+
                     |
                     )||(           ← coupled transformer
                     )|| L_m       (primary inductance L_m,
                     )||(            turns ratio n = N_s/N_p)
                     |
                    gnd_pri

                     |          isolated  |
   gnd_pri          (||)          gap     |          gnd_sec
                    (||) L_s = L_m·n²              |
                    (||)                          (|)
                     |    D (anode toward sec      |
                     +--+         neg, cath toward |
                        |         output)          |
                        +--+--+--+--- V_o (neg)    |
                        |  |  |  |                 |
                        C  R  load                 |
                        |  |  |  |                 |
                      gnd_sec  gnd_sec
```

- **Primary side** (left of the transformer): input bus $V_g$, switch
  $S$, primary winding (magnetizing inductance $L_m$, $N_p$ turns).
- **Secondary side** (right of the transformer, galvanically isolated):
  secondary winding ($N_s$ turns), diode $D$ in series, output cap
  $C$ and load $R$ in parallel.
- Turns ratio: $n = N_s / N_p$.

### Switching intervals

- **ON**: $S$ closed → primary winding sees $V_g$, energy builds up
  in the magnetizing field ($i_{L_m}$ grows linearly). The secondary
  winding sees $n \\cdot v_p = n V_g$ in the "wrong direction" relative
  to $V_o$ → diode reverse-biased, $i_{sec} = 0$. The cap discharges
  through the load.

- **OFF**: $S$ opens. The magnetizing field must continue (flux
  continuity). The current that was flowing on the primary
  ($i_{L_m}$) **transfers** to the secondary as $i_{L_m} / n$,
  forward-biases the diode, and pumps charge into the cap.


## 2. Switched (instantaneous) model

State variables: primary magnetizing current $i_{L_m}$ and secondary
output voltage magnitude $v_o$.

### 2.1 ON interval ($S$ closed, $D$ off)

$$
L_m \\cdot \\frac{di_{L_m}}{dt} = v_g
\\qquad
C \\cdot \\frac{dv_o}{dt} = -\\frac{v_o}{R}
$$

The primary sees the full input bus; the secondary cap drains.

### 2.2 OFF interval ($S$ open, $D$ on)

The secondary winding clamps to $v_o + V_F \\approx v_o$ (ideal diode
forward drop ≈ 0). The primary sees the *reflected* secondary
voltage:

$$
v_p \\big|_{OFF} = \\frac{v_o}{n}
$$

so the magnetizing field discharges:

$$
L_m \\cdot \\frac{di_{L_m}}{dt} = -\\frac{v_o}{n}
\\qquad
C \\cdot \\frac{dv_o}{dt} = \\frac{i_{L_m}}{n} - \\frac{v_o}{R}
$$

The $1/n$ factors are the transformer's only contribution to the
math — everything else is identical to the buck-boost.


## 3. State-space averaging

Average $q \\to d$:

$$\\boxed{
\\;\\; L_m \\frac{di_{L_m}}{dt} = d v_g - (1-d) \\frac{v_o}{n}
\\;\\;}
$$

$$\\boxed{
\\;\\; C \\frac{dv_o}{dt} = (1-d) \\frac{i_{L_m}}{n} - \\frac{v_o}{R}
\\;\\;}
$$

Pattern check: compare to the buck-boost's average model in
`buck_boost_model.py`. The flyback's equations have $1/n$ wherever
the buck-boost's had $1$. Set $n = 1$ and the two are identical.

### 3.1 Steady-state

$$
0 = D V_g - (1-D) \\frac{V_o}{n}
\\;\\implies\\;
\\boxed{V_o = n \\cdot V_g \\cdot \\frac{D}{1 - D}}
$$

$$
0 = (1-D) \\frac{I_{L_m}}{n} - \\frac{V_o}{R}
\\;\\implies\\;
I_{L_m} = \\frac{n V_o}{R (1-D)}
$$

For $n = 1$ this is the buck-boost. The turns ratio gives the designer
a knob to set the operating $D$: for any desired $V_o / V_g$, pick
$n$ such that $D$ lands in a comfortable range (typically 0.3–0.6).


In [ ]:
params = FlybackParams()
print(operating_point_report(params))


## 4. Small-signal linearization

Perturb and drop products:

$$
L_m \\frac{d\\hat i_{L_m}}{dt}
   = D \\hat v_g + \\left(V_g + \\frac{V_o}{n}\\right) \\hat d
   - \\frac{(1-D)}{n} \\hat v_o
$$

$$
C \\frac{d\\hat v_o}{dt}
   = \\frac{(1-D)}{n} \\hat i_{L_m} - \\frac{I_{L_m}}{n} \\hat d
   - \\frac{\\hat v_o}{R}
$$

The $-I_{L_m}/n \\cdot \\hat d$ term in the cap equation is what gives
the flyback its RHP zero — same mechanism as the buck-boost.


## 5. State-space matrices

$$
A = \\begin{bmatrix}
0 & -(1-D)/(n L_m) \\\\
(1-D)/(n C) & -1/(R C)
\\end{bmatrix}
$$

$$
B = \\begin{bmatrix}
(V_g + V_o/n)/L_m & D/L_m \\\\
-I_{L_m}/(n C) & 0
\\end{bmatrix}
$$

Set $n = 1$ to recover the buck-boost matrices verbatim.


In [ ]:
A, B, C_mat, D_mat = flyback_state_space(params)
print("A ="); print(A); print()
print("B = [col 0: d̂   col 1: v̂_g]"); print(B); print()
print("C =", C_mat)
print("D (feedthrough) =", D_mat)
print()
eig = np.linalg.eigvals(A)
print(f"A eigenvalues: {eig}")
print(f"Pole magnitude (= ω_n): {abs(eig[0]):.1f} rad/s  "
      f"(expect {params.omega_n:.1f})")


## 6. Transfer functions

### 6.1 $G_{vd}(s)$

Closed form on the secondary side:

$$
G_{vd}(s) = \\frac{n V_g}{(1-D)^2} \\cdot
\\frac{1 - s/\\omega_{z,RHP}}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2}
$$

with:

| Quantity | Flyback | Buck-boost |
|---|---|---|
| DC gain | $n V_g / (1-D)^2$ | $V_g / (1-D)^2$ |
| $\\omega_n$ | $(1-D)/(n \\sqrt{L_m C})$ | $(1-D)/\\sqrt{L C}$ |
| $Q$ | $(1-D) R \\sqrt{C/L_m} / n$ | $(1-D) R \\sqrt{C/L}$ |
| $\\omega_{z,RHP}$ | $R(1-D)^2 / (n^2 L_m D)$ | $R(1-D)^2 / (L D)$ |

The $1/n^2$ multiplier on the RHP zero is the big design lever — a
1:0.5 step-down transformer (n = 0.5) **quadruples** the RHP zero
frequency, giving 4× the closed-loop bandwidth headroom.

### 6.2 $G_{vg}(s)$, $Z_{out}(s)$

$$
G_{vg}(s) = \\frac{n D / (1-D)}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2}
$$

$$
Z_{out}(s) = \\frac{s L_m / (n^2 (1-D)^2)}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2}
$$

Same denominator everywhere. No RHP zeros in these (only $G_{vd}$
has it).


In [ ]:
Gvd = control_to_output_tf(params)
Gvg = line_to_output_tf(params)
Zout = output_impedance_tf(params)

print(f"Gvd(0)  = {Gvd.num[1] / Gvd.den[2]:.3f} V/duty  "
      f"(expect n·V_g/(1-D)² = {params.n*params.V_g/(1-params.D)**2:.3f})")
print(f"Gvg(0)  = {Gvg.num[0] / Gvg.den[2]:.4f} V/V    "
      f"(expect n·D/(1-D) = {params.n*params.D/(1-params.D):.4f})")
print()
print(f"Gvd zero: {np.roots(Gvd.num)} (expect RHP at +{params.omega_z_rhp:.0f})")
print(f"Gvd poles: {np.roots(Gvd.den)}")


### 6.3 Bode plots

Compare $G_{vd}$ (which has the RHP zero) with $G_{vg}$ and
$Z_{out}$ (which don't). Mark the LC pole and RHP zero positions.


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for tf, name, style in [
    (Gvd,  r"$G_{vd}$ control → output",   "-"),
    (Gvg,  r"$G_{vg}$ line → output",      "--"),
    (Zout, r"$Z_{out}$ load → output",     ":"),
]:
    _, mag, ph = signal.bode(tf, w=w)
    ax_mag.semilogx(f, mag, style, label=name)
    ax_ph.semilogx(f, ph, style, label=name)

for ax in (ax_mag, ax_ph):
    ax.axvline(params.f_n,     color="C0", linestyle=":", alpha=0.4,
               label=f"$f_n$ = {params.f_n:.0f} Hz")
    ax.axvline(params.f_z_rhp, color="C3", linestyle=":", alpha=0.6,
               label=f"$f_{{z,RHP}}$ = {params.f_z_rhp:.0f} Hz")
    ax.legend(loc="best", fontsize=8)

ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(f"Flyback open-loop "
                 f"($V_g$={params.V_g}V → $V_o$={params.V_o}V, "
                 f"n={params.n}, D={params.D:.2f})")
plt.tight_layout()
plt.show()


## 7. The wrong-way step response

Same RHP-zero signature as the buck-boost: positive duty step causes
$v_o$ to **dip** before rising. The mechanism is identical
(magnetizing current must build before secondary current can deliver
charge), just routed through the transformer.


In [ ]:
duty_step = 0.01
t = np.linspace(0, 5e-3, 5000)
_, y_step = signal.step(Gvd, T=t)
v_o_pred = params.V_o + duty_step * y_step

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t * 1e3, v_o_pred, label="$v_o$ (analytical small-signal)")
ax.axhline(params.V_o, color="k", linestyle=":", alpha=0.4,
           label=f"Pre-step $V_o$ = {params.V_o} V")
expected_new = params.V_o + duty_step * params.n * params.V_g / (1 - params.D)**2
ax.axhline(expected_new, color="g", linestyle=":", alpha=0.5,
           label=f"Predicted new $V_o$ ≈ {expected_new:.3f} V")
ax.set_xlabel("Time [ms]")
ax.set_ylabel("$v_o$ [V]")
ax.set_title(f"Flyback step response to a +{duty_step*100:.0f}% duty step "
             "(dip from RHP zero)")
ax.legend()
plt.tight_layout()
plt.show()

dip = params.V_o - np.min(v_o_pred)
print(f"Pre-step V_o     = {params.V_o:.4f} V")
print(f"Initial dip      = {dip * 1e3:.1f} mV "
      f"({dip / params.V_o * 100:.2f} % of V_o)")
print(f"Final v_o        = {v_o_pred[-1]:.4f} V")


## 8. Self-consistency checks

Same three rigorous checks: poles, DC gains, and ss2tf round-trip.


In [ ]:
Gvd_closed = control_to_output_tf(params)

# (1) Poles
ss_poles = sorted(np.linalg.eigvals(A), key=lambda z: z.imag)
tf_poles = sorted(np.roots(Gvd_closed.den), key=lambda z: z.imag)
print("(1) Poles:")
print(f"    SS:  {ss_poles}")
print(f"    TF:  {tf_poles}")
pole_match = np.allclose(ss_poles, tf_poles, rtol=1e-10)
print(f"    → match: {pole_match}")

# (2) DC gains
print()
print("(2) DC gains:")
print(f"    Gvd(0)  = {Gvd_closed.num[1] / Gvd_closed.den[2]:8.4f}  "
      f"(expect n·V_g/(1-D)² = {params.n*params.V_g/(1-params.D)**2:.4f})")
Gvg_local = line_to_output_tf(params)
print(f"    Gvg(0)  = {Gvg_local.num[0] / Gvg_local.den[2]:8.4f}  "
      f"(expect n·D/(1-D) = {params.n*params.D/(1-params.D):.4f})")

# (3) ss2tf round-trip
num_from_ss, den_from_ss = signal.ss2tf(A, B, C_mat, D_mat, input=0)
num_from_ss = np.trim_zeros(num_from_ss.flatten(), trim='f')
scale_ss = den_from_ss[0]
scale_cf = Gvd_closed.den[0]
num_ss_norm = num_from_ss / scale_ss
num_cf_norm = np.array(Gvd_closed.num) / scale_cf
den_ss_norm = np.array(den_from_ss) / scale_ss
den_cf_norm = np.array(Gvd_closed.den) / scale_cf
print()
print("(3) ss2tf round-trip:")
print(f"    Closed-form num = {num_cf_norm}")
print(f"    From-SS num     = {num_ss_norm}")
print(f"    Closed-form den = {den_cf_norm}")
print(f"    From-SS den     = {den_ss_norm}")
round_trip_ok = (
    np.allclose(num_ss_norm, num_cf_norm, rtol=1e-9)
    and np.allclose(den_ss_norm, den_cf_norm, rtol=1e-9)
)
print(f"    → match: {round_trip_ok}")

assert pole_match and round_trip_ok, "Self-consistency check failed!"
print()
print("✅  All three self-consistency checks pass.")


## 9. The "n picks D" design lever

For a fixed $V_g$ and $V_o$, sweep the turns ratio $n$ and watch how
the operating $D$ moves. The design rule: pick $n$ so that $D$ falls
in $[0.3, 0.6]$ — that gives margin for line/load variation without
hitting the extremes (where the RHP zero collapses for high $D$, or
the cap ripple blows up for low $D$).


In [ ]:
n_grid = np.linspace(0.1, 2.0, 100)
D_grid = []
fz_grid = []
fn_grid = []
for n_val in n_grid:
    # D = V_o / (V_o + n·V_g)
    D = params.V_o / (params.V_o + n_val * params.V_g)
    D_grid.append(D)
    # f_z = R·(1-D)²/(n²·L_m·D · 2π)
    fz = params.R * (1-D)**2 / (n_val**2 * params.L_m * D) / (2*np.pi)
    fz_grid.append(fz)
    fn = (1-D) / (n_val * np.sqrt(params.L_m * params.C)) / (2*np.pi)
    fn_grid.append(fn)

fig, (ax_d, ax_f) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_d.plot(n_grid, D_grid, "C0", linewidth=2)
ax_d.axhline(0.3, color="g", linestyle=":", alpha=0.4, label="D = 0.3 (lower bound)")
ax_d.axhline(0.6, color="g", linestyle=":", alpha=0.4, label="D = 0.6 (upper bound)")
ax_d.set_ylabel("Operating duty D")
ax_d.set_title(f"Flyback: choosing $n$ to land $D$ in the comfort zone "
               f"($V_g$={params.V_g}V, $V_o$={params.V_o}V)")
ax_d.legend()

ax_f.semilogy(n_grid, fz_grid, "C3", linewidth=2, label=r"$f_{z,RHP}$")
ax_f.semilogy(n_grid, fn_grid, "C0", linestyle="--", label=r"$f_n$")
ax_f.set_xlabel("Turns ratio n = $N_s / N_p$")
ax_f.set_ylabel("Frequency [Hz]")
ax_f.legend()
plt.tight_layout()
plt.show()

# Find the n that gives D = 0.5
n_at_D05 = params.V_o / params.V_g
print(f"For V_o={params.V_o}V from V_g={params.V_g}V at D=0.5: "
      f"n must be {n_at_D05:.3f}")
print(f"  At this n: f_z_RHP = {fz_grid[np.argmin(np.abs(n_grid - n_at_D05))]:.0f} Hz")


## 10. Summary

The flyback is the buck-boost with a transformer instead of a single
inductor. The math derivation is identical except for the $1/n$
factors on the reflected secondary side. The RHP zero is at a
**higher** frequency than the buck-boost at the same $L_m, C, R, D$
(by $1/n^2$) — so the loop can run faster, all else equal.

Math validated by three checks (poles, DC gains, ss2tf round-trip).

**Cross-validation against Pulsim**: open
[`00_flyback_pulsim_validation.ipynb`](00_flyback_pulsim_validation.ipynb)
for an executed notebook that builds the flyback (two-winding linear
transformer + rectifier diode + output cap) in Pulsim and overlays
the steady-state output voltage.

**Next**: open `02_flyback_controller.ipynb` to size a Type-III
compensator and run a switched closed-loop simulation.

**Suggested exercises**

1. Vary $n$ from 0.2 to 2.0 and re-derive the operating point and
   RHP zero. Where does $f_z$ go?
2. Add a non-ideal transformer (leakage inductance $L_\\ell$) to the
   model. Where does $L_\\ell$ enter the small-signal $G_{vd}(s)$?
   (Hint: it adds a fast-pole right-half-plane pair that's usually
   damped by the snubber.)
3. Derive the forward converter (buck with isolation). How does
   the tertiary reset winding affect the average model?
